# Phase 1: Data Loading & Preprocessing

**Project:** Explaining Spatio-Temporal Clustering with ClAMP  
**Dataset:** Hofer LandBus — ~63,000 mobility records (Germany)  
**Goal:** Load, clean, and engineer features for spatio-temporal clustering.

> ⚠️ **Data Notice:** The raw Hofer LandBus dataset is confidential.  
> To reproduce this notebook, replace the data loading cell with your own DataFrame  
> that matches the expected schema described below.

### Expected Input Schema
| Column | Type | Description |
|---|---|---|
| `departure_time` | datetime | Timestamp of trip departure |
| `arrival_time` | datetime | Timestamp of trip arrival |
| `departure_lat` | float | Departure latitude (EPSG:4326) |
| `departure_lon` | float | Departure longitude (EPSG:4326) |
| `destination_lat` | float | Arrival latitude |
| `destination_lon` | float | Arrival longitude |

### Outputs
- `prepare_hofer_data` — full dataset with engineered time features
- `prepare_hofer_departure` — start-point DataFrame for clustering
- `prepare_hofer_destination` — end-point DataFrame for clustering

## 1.1 Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
from shapely.geometry import Point
import gc

## 1.2 Data Loading

> Replace this cell with your own data source.  
> The cell below shows the expected structure using a synthetic sample.

In [ ]:
# ---- REPLACE THIS BLOCK WITH YOUR DATA LOADING ----
# Example: load from CSV
# prepare_hofer_data = pd.read_csv('data/hofer_landbus.csv', parse_dates=['departure_time', 'arrival_time'])

# Synthetic schema preview (for reference only)
schema_preview = pd.DataFrame({
    'departure_time':  pd.to_datetime(['2023-01-01 08:00', '2023-01-01 08:15']),
    'arrival_time':    pd.to_datetime(['2023-01-01 08:30', '2023-01-01 08:45']),
    'departure_lat':   [50.3133, 50.3200],
    'departure_lon':   [11.9078, 11.9150],
    'destination_lat': [50.3300, 50.3400],
    'destination_lon': [11.9200, 11.9300],
})
print("Expected schema:")
schema_preview.head()

## 1.3 Feature Engineering — Temporal

In [ ]:
# Convert departure and arrival times to seconds since midnight
# This enables temporal distance calculations in ST-DBSCAN

prepare_hofer_data['start_time_seconds'] = (
    prepare_hofer_data['departure_time'].dt.hour * 3600 +
    prepare_hofer_data['departure_time'].dt.minute * 60 +
    prepare_hofer_data['departure_time'].dt.second
)

prepare_hofer_data['end_time_seconds'] = (
    prepare_hofer_data['arrival_time'].dt.hour * 3600 +
    prepare_hofer_data['arrival_time'].dt.minute * 60 +
    prepare_hofer_data['arrival_time'].dt.second
)

prepare_hofer_data['start_hour'] = prepare_hofer_data['departure_time'].dt.hour

print(f"Dataset shape: {prepare_hofer_data.shape}")
prepare_hofer_data.head(2)

## 1.4 Time Scaling for Spatial Equivalence

ST-DBSCAN requires both spatial (meters) and temporal (seconds) distances.  
We scale time so that **3600 seconds (1 hour) ≈ 250 meters** in the combined distance space.

In [ ]:
# Scale factor: 1 hour -> 250 metres equivalent
TIME_SCALE_FACTOR = 250 / 3600  # ≈ 0.0694

prepare_hofer_data['time_scaled'] = prepare_hofer_data['start_time_seconds'] * TIME_SCALE_FACTOR

print(f"Time scale factor: {TIME_SCALE_FACTOR:.4f} (1 hour = 250m equivalent)")

## 1.5 Prepare Departure & Destination DataFrames

In [ ]:
# Start points — used for departure clustering
prepare_hofer_departure = prepare_hofer_data[[
    'departure_lon', 'departure_lat', 'start_time_seconds', 'time_scaled'
]].copy().reset_index(drop=True)

# End points — used for destination clustering
prepare_hofer_destination = prepare_hofer_data[[
    'destination_lon', 'destination_lat', 'end_time_seconds'
]].copy().reset_index(drop=True)

print(f"Departure points:    {len(prepare_hofer_departure):,}")
print(f"Destination points:  {len(prepare_hofer_destination):,}")

## 1.6 Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Spatial distribution of departure points
axes[0].scatter(
    prepare_hofer_departure['departure_lon'],
    prepare_hofer_departure['departure_lat'],
    s=1, alpha=0.3, color='steelblue'
)
axes[0].set_title('Departure Points — Spatial Distribution')
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
axes[0].grid(True)

# Temporal distribution
axes[1].hist(prepare_hofer_data['start_hour'], bins=24, color='coral', edgecolor='white')
axes[1].set_title('Departure Hour Distribution')
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Number of Trips')
axes[1].grid(True, axis='y')

plt.tight_layout()
plt.savefig('../results/01_eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: results/01_eda_overview.png")

In [ ]:
# Summary statistics
print("=== Dataset Summary ===")
print(f"Total records:         {len(prepare_hofer_data):,}")
print(f"Date range:            {prepare_hofer_data['departure_time'].min().date()} → {prepare_hofer_data['departure_time'].max().date()}")
print(f"Lat range (departure): {prepare_hofer_data['departure_lat'].min():.4f} → {prepare_hofer_data['departure_lat'].max():.4f}")
print(f"Lon range (departure): {prepare_hofer_data['departure_lon'].min():.4f} → {prepare_hofer_data['departure_lon'].max():.4f}")
print(f"Missing values:\n{prepare_hofer_data.isnull().sum()[prepare_hofer_data.isnull().sum() > 0]}")